# Sinais de aumento incomum

## Objetivo

Implementar uma regra inicial, explícita e reproduzível para identificar
semanas em que a quantidade observada de registros de SRAG em residentes
de Minas Gerais apresenta aumento incomum em relação à referência histórica.

Cada avaliação deve registrar:

- região;
- ano e semana epidemiológica;
- período correspondente;
- valor observado;
- referência histórica;
- regra aplicada;
- resultado da avaliação;
- justificativa do sinal;
- disponibilidade suficiente ou insuficiente de dados.

A regra implementada nesta etapa possui finalidade operacional e
exploratória. Ela não constitui, isoladamente, um limiar epidemiológico
validado de surto ou emergência.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent

current_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_mg_weekly_coverage.csv"
)

reference_path = (
    project_root
    / "data"
    / "processed"
    / "srag_mg_historical_reference_2019_2025.csv"
)

current = pd.read_csv(
    current_path,
    sep=";",
)

reference = pd.read_csv(
    reference_path,
    sep=";"
)

print(f"Série corrente: {len(current)} semanas")
print(f"Referência histórica: {len(reference)} semanas")

Série corrente: 52 semanas
Referência histórica: 52 semanas


In [2]:
assert len(current) == 52
assert len(reference) == 52

assert current["epi_week"].is_unique
assert reference["epi_week"].is_unique

assert set(range(1, 53)) == set(current["epi_week"])
assert set(range(1, 53)) == set(reference["epi_week"])

required_reference_columns = {
    "epi_week",
    "historical_median",
    "q25",
    "q75",
    "historical_years",
    "has_minimum_history",
}

assert required_reference_columns.issubset(reference.columns)

print("✓ Entradas validadas.")

✓ Entradas validadas.


In [4]:
signal_evaluation = current.merge(
    reference[
        [
            "epi_week",
            "historical_median",
            "q25",
            "q75",
            "historical_years",
            "has_minimum_history",
        ]
    ],
    on="epi_week",
    how="left",
    validate="one_to_one",
)

signal_evaluation["region"] = "MG"

signal_evaluation = signal_evaluation[
    [
        "region",
        "epi_year",
        "epi_week",
        "week_start",
        "week_end",
        "record_count",
        "historical_median",
        "q25",
        "q75",
        "historical_years",
        "has_minimum_history",
        "calendar_status",
        "data_status",
    ]
]

display(signal_evaluation.head())

,region,epi_year,epi_week,week_start,week_end,record_count,historical_median,q25,q75,historical_years,has_minimum_history,calendar_status,data_status
0,MG,2026,1,2026-01-04,2026-01-10,436.0,397.0,169.0,2291.5,7,True,encerrada,observado
1,MG,2026,2,2026-01-11,2026-01-17,415.0,422.0,162.0,2537.0,7,True,encerrada,observado
2,MG,2026,3,2026-01-18,2026-01-24,376.0,413.0,196.5,2288.5,7,True,encerrada,observado
3,MG,2026,4,2026-01-25,2026-01-31,367.0,412.0,161.5,2115.5,7,True,encerrada,observado
4,MG,2026,5,2026-02-01,2026-02-07,388.0,363.0,206.0,1910.5,7,True,encerrada,observado


## Regra inicial de aumento incomum

Uma semana observada é classificada como `SIGNAL` quando atende
simultaneamente às seguintes condições:

1. existem dados observados para a semana;
2. existem pelo menos 3 anos históricos válidos;
3. a contagem observada é superior ao percentil 75 histórico da mesma
   semana epidemiológica;
4. a contagem observada é pelo menos 50% superior à mediana histórica
   da mesma semana.

Formalmente:

`record_count > Q75`

e

`record_count >= historical_median × 1.5`

Caso não exista observação suficiente da série corrente ou histórico
mínimo, a avaliação recebe `INSUFFICIENT_DATA`, e não `NO_SIGNAL`.

O fator de 1,5 é um parâmetro operacional inicial para esta implementação.
Ele não representa um limiar epidemiológico validado.

In [7]:
SIGNAL_FACTOR = 1.5

signal_evaluation["signal_status"] = "NO_SIGNAL"

insufficient_data = (
    signal_evaluation["record_count"].isna()
    | ~signal_evaluation["has_minimum_history"]
    | signal_evaluation["historical_median"].isna()
    | signal_evaluation["q75"].isna()
)

signal_evaluation.loc[
    insufficient_data,
    "signal_status",
] = "INSUFFICIENT_DATA"

unusual_increase = (
    ~insufficient_data
    & (
        signal_evaluation["record_count"]
        > signal_evaluation["q75"]
    )
    & (
        signal_evaluation["record_count"]
        >= (
            signal_evaluation["historical_median"]
            * SIGNAL_FACTOR
        )
    )
)

signal_evaluation.loc[
    unusual_increase,
    "signal_status",
] = "SIGNAL"

In [8]:
signal_evaluation["median_threshold"] = (
    signal_evaluation["historical_median"]
    * SIGNAL_FACTOR
)

signal_evaluation["ratio_to_median"] = (
    signal_evaluation["record_count"]
    / signal_evaluation["historical_median"]
)

signal_evaluation["above_q75"] = (
    signal_evaluation["record_count"]
    > signal_evaluation["q75"]
)

signal_evaluation["above_factor_threshold"] = (
    signal_evaluation["record_count"]
    >= signal_evaluation["median_threshold"]
)

In [9]:
for column in [
    "above_q75",
    "above_factor_threshold",
]:
    signal_evaluation[column] = (
        signal_evaluation[column].astype("boolean")
    )

    signal_evaluation.loc[
        insufficient_data,
        column,
    ] = pd.NA

In [10]:
def explain_signal(row):
    if row["signal_status"] == "INSUFFICIENT_DATA":
        if pd.isna(row["record_count"]):
            return (
                "Dados observados indisponíveis para esta semana."
            )
        
        if not row["has_minimum_history"]:
            return (
                "Histórico insuficiente para aplicar a regra."
            )
        
        return (
            "Referência histórica indisponível para esta semana."
        )
    
    if row["signal_status"] == "SIGNAL":
        return (
            f"Observado ({int(row['record_count'])}) acima de Q75 "
            f"({row['q75']:.1f}) e pelo menos "
            f"{SIGNAL_FACTOR:.1}x a mediana histórica "
            f"({row['historical_median']:.1f})."
        )
    
    return (
        "Critérios de aumento incomum não atendidos."
    )


signal_evaluation["signal_reason"] = (
    signal_evaluation.apply(
        explain_signal,
        axis=1,
    )
)

In [11]:
display(
    signal_evaluation[
        "signal_status"
    ].value_counts(dropna=False)
)

display(
    signal_evaluation.loc[
        signal_evaluation["signal_status"] == "SIGNAL",
        [
            "region",
            "epi_year",
            "epi_week",
            "week_start",
            "week_end",
            "record_count",
            "historical_median",
            "q75",
            "median_threshold",
            "ratio_to_median",
            "signal_reason",
        ]
    ]
)

signal_status
NO_SIGNAL            37
INSUFFICIENT_DATA    14
SIGNAL                1
Name: count, dtype: int64

,region,epi_year,epi_week,week_start,week_end,record_count,historical_median,q75,median_threshold,ratio_to_median,signal_reason
14,MG,2026,15,2026-04-12,2026-04-18,1066.0,698.0,933.5,1047.0,1.527221,Observado (1066) acima de Q75 (933.5) e pelo m...


In [12]:
display(
    signal_evaluation.loc[
        signal_evaluation["epi_week"].isin([38, 39, 40]),
        [
            "epi_week",
            "record_count",
            "data_status",
            "signal_status",
            "signal_reason",
        ],
    ]
)

,epi_week,record_count,data_status,signal_status,signal_reason
37,38,102.0,observado,NO_SIGNAL,Critérios de aumento incomum não atendidos.
38,39,NaN,sem_cobertura_confirmada,INSUFFICIENT_DATA,Dados observados indisponíveis para esta semana.
39,40,NaN,sem_cobertura_confirmada,INSUFFICIENT_DATA,Dados observados indisponíveis para esta semana.


In [14]:
assert signal_evaluation["epi_week"].is_unique
assert len(signal_evaluation) == 52

observed = signal_evaluation.loc[
    signal_evaluation["data_status"] == "observado"
]

assert observed["record_count"].notna().all()

uncovered = signal_evaluation.loc[
    signal_evaluation["data_status"]
    == "sem_cobertura_confirmada"
]

assert uncovered["record_count"].isna().all()

assert uncovered[
    "signal_status"
].eq("INSUFFICIENT_DATA").all()

signals = signal_evaluation.loc[
    signal_evaluation["signal_status"] == "SIGNAL"
]

assert signals["above_q75"].eq(True).all()
assert signals["above_factor_threshold"].eq(True).all()
assert signals["signal_reason"].notna().all()

print("✓ Avaliação dos sinais validada.")
print(f"Semanas avaliadas: {len(signal_evaluation)}")
print(f"Semanas observadas: {len(observed)}")
print(f"Sinais gerados: {len(signals)}")
print(f"Dados insuficientes: {len(uncovered)}")

✓ Avaliação dos sinais validada.
Semanas avaliadas: 52
Semanas observadas: 38
Sinais gerados: 1
Dados insuficientes: 14


In [15]:
signals_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_mg_signal_evaluation.csv"
)

signal_evaluation.to_csv(
    signals_path,
    sep=";",
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d",
    na_rep="",
)

print(f"Avaliação salva em: {signals_path}")

Avaliação salva em: /home/matheuspessoa/Sentinela/data/processed/srag_2026_mg_signal_evaluation.csv


In [17]:
exported_signals = pd.read_csv(
    signals_path,
    sep=";"
)

assert len(exported_signals) == 52
assert exported_signals["epi_week"].is_unique

assert (
    exported_signals["signal_status"]
    == "SIGNAL"
).sum() == len(signals)

assert (
    exported_signals["signal_status"]
    == "INSUFFICIENT_DATA"
).sum() == len(uncovered)

exported_signal_rows = exported_signals.loc[
    exported_signals["signal_status"] == "SIGNAL"
]

exported_signal_rows["signal_reason"].notna().all()

print("✓ Exportação dos sinais validada.")

✓ Exportação dos sinais validada.


In [18]:
trace_columns = [
    "region",
    "epi_year",
    "epi_week",
    "week_start",
    "week_end",
    "record_count",
    "historical_median",
    "q75",
    "median_threshold",
    "ratio_to_median",
    "historical_years",
    "signal_status",
    "signal_reason",
]

display(
    signal_evaluation.loc[
        signal_evaluation["signal_status"] == "SIGNAL",
        trace_columns,
    ]
)

,region,epi_year,epi_week,week_start,week_end,record_count,historical_median,q75,median_threshold,ratio_to_median,historical_years,signal_status,signal_reason
14,MG,2026,15,2026-04-12,2026-04-18,1066.0,698.0,933.5,1047.0,1.527221,7,SIGNAL,Observado (1066) acima de Q75 (933.5) e pelo m...


## Regra V1 de sinais de aumento incomum

### Regra

Para cada região e semana epidemiológica com dados suficientes, a
contagem observada é comparada com a referência histórica da mesma semana.

Um `SIGNAL` é gerado somente quando as duas condições abaixo são
satisfeitas simultaneamente:

1. `record_count > Q75`
2. `record_count >= historical_median × 1.5`

O fator `1.5` representa um aumento de pelo menos 50% em relação à
mediana histórica.

### Estados possíveis

- `SIGNAL`: todos os critérios da regra foram satisfeitos;
- `NO_SIGNAL`: existem dados suficientes, mas pelo menos um dos critérios
  de aumento incomum não foi satisfeito;
- `INSUFFICIENT_DATA`: não existem dados suficientes para aplicar a regra.

`INSUFFICIENT_DATA` não deve ser interpretado como ausência de aumento.

### Referência utilizada

A referência histórica utiliza os anos de 2019 a 2025 e é calculada
separadamente para cada semana epidemiológica.

São utilizados:

- mediana histórica;
- percentil 75 (Q75);
- quantidade de anos históricos disponíveis.

A série corrente de 2026 não participa da construção de sua própria
referência.

### Rastreabilidade

Cada avaliação registra:

- região;
- ano epidemiológico;
- semana epidemiológica;
- início e fim da semana;
- contagem observada;
- mediana histórica;
- Q75 histórico;
- limiar correspondente a 1,5 vezes a mediana;
- razão entre observado e mediana;
- quantidade de anos históricos;
- resultado da avaliação;
- justificativa textual da classificação.

Dessa forma, cada sinal pode ser reproduzido a partir dos valores
armazenados e da regra documentada.

### Limitações

A regra V1 é operacional e exploratória.

O fator de 1,5 não representa um limiar epidemiológico validado de surto,
emergência ou anomalia.

Q75 é utilizado como componente da comparação histórica, mas não deve ser
interpretado isoladamente como limite epidemiológico de alerta.

A referência histórica contém períodos com comportamentos epidemiológicos
distintos e pode ser afetada por mudanças na vigilância, diagnóstico,
notificação e qualidade dos dados.

Os sinais devem ser interpretados como indicação de aumento incomum segundo
a regra implementada, e não como confirmação de um evento epidemiológico.